# Unit 2.2 The LlamaIndex framework

In [1]:
# Añade el nombre de un ejercicio después de preparar sus requisitos.
external_exercises = set()

def install_packages(*args):
    import subprocess
    import sys

    subprocess.run([sys.executable, "-m", "pip", "install", *args], check=True)


## Introduction to LlamaHub

### Paquetes y componentes

Relaciona cada componente con su paquete de integración.

In [2]:
integrations = {
    "llm": "llama-index-llms-huggingface-api",
    "embedding": "llama-index-embeddings-huggingface",
    "vector_store": "llama-index-vector-stores-chroma",
}
assert len(set(integrations.values())) == len(integrations)
for component, package in integrations.items():
    print(f"{component} → {package}")


llm → llama-index-llms-huggingface-api
embedding → llama-index-embeddings-huggingface
vector_store → llama-index-vector-stores-chroma


In [3]:
if "llamahub" in external_exercises:
    # Substitute the two placeholders before installing.
    # %pip install llama-index-{component-type}-{framework-name}
    pass


In [4]:
if "llamahub" in external_exercises:
    install_packages(*['llama-index-llms-huggingface-api', 'llama-index-embeddings-huggingface'])


In [5]:
if "llamahub" in external_exercises:
    from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
    import os
    from dotenv import load_dotenv

    # Load the .env file
    load_dotenv()

    # Retrieve HF_TOKEN from the environment variables
    hf_token = os.getenv("HF_TOKEN")

    llm = HuggingFaceInferenceAPI(
        model_name="Qwen/Qwen2.5-Coder-32B-Instruct",
        temperature=0.7,
        max_tokens=100,
        token=hf_token,
        provider="auto"
    )

    response = llm.complete("Hello, how are you?")
    print(response)
    # I am good, how can I help you today?


### Ingesta y recuperación local

Divide documentos en fragmentos y devuelve el fragmento relevante junto con su fuente.

In [6]:
import re

documents = {
    "manual.txt": "El teléfono requiere un cargador USB-C. El brillo se ajusta desde Configuración.",
    "soporte.txt": "La garantía cubre fallos de fabricación. Para reparaciones, contacta con soporte.",
}

def tokens(text):
    return set(re.findall(r"[\w-]+", text.casefold()))

nodes = [
    {"source": filename, "text": sentence.strip()}
    for filename, content in documents.items()
    for sentence in content.split(".")
    if sentence.strip()
]

def retrieve(query):
    query_tokens = tokens(query)
    scored = [(len(query_tokens & tokens(node["text"])), node) for node in nodes]
    return [node for score, node in sorted(scored, key=lambda pair: -pair[0]) if score]

matches = retrieve("cargador")
assert [node["source"] for node in matches] == ["manual.txt"]
assert retrieve("astronomía") == []
for node in matches:
    print(f"[{node['source']}] {node['text']}")


[manual.txt] El teléfono requiere un cargador USB-C


In [7]:
if "components" in external_exercises:
    from llama_index.core import SimpleDirectoryReader

    reader = SimpleDirectoryReader(input_dir="path/to/directory")
    documents = reader.load_data()


In [8]:
if "components" in external_exercises:
    from llama_index.core import Document
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
    from llama_index.core.node_parser import SentenceSplitter
    from llama_index.core.ingestion import IngestionPipeline

    # create the pipeline with transformations
    pipeline = IngestionPipeline(
        transformations=[
            SentenceSplitter(chunk_overlap=0),
            HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
        ]
    )

    nodes = await pipeline.arun(documents=[Document.example()])


In [9]:
if "components" in external_exercises:
    install_packages(*['llama-index-vector-stores-chroma'])


In [10]:
if "components" in external_exercises:
    import chromadb
    from llama_index.vector_stores.chroma import ChromaVectorStore

    db = chromadb.PersistentClient(path="./alfred_chroma_db")
    chroma_collection = db.get_or_create_collection("alfred")
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

    pipeline = IngestionPipeline(
        transformations=[
            SentenceSplitter(chunk_size=25, chunk_overlap=0),
            HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
        ],
        vector_store=vector_store,
    )


In [11]:
if "components" in external_exercises:
    from llama_index.core import VectorStoreIndex
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding

    embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
    index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)


In [12]:
if "components" in external_exercises:
    from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

    llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")
    query_engine = index.as_query_engine(
        llm=llm,
        response_mode="tree_summarize",
    )
    query_engine.query("What is the meaning of life?")
    # The meaning of life is 42


In [13]:
if "components" in external_exercises:
    from llama_index.core.evaluation import FaithfulnessEvaluator

    # Reuse query_engine from the previous section.
    # Reuse llm from the previous section.

    # query index
    evaluator = FaithfulnessEvaluator(llm=llm)
    response = query_engine.query(
        "What battles took place in New York City in the American Revolution?"
    )
    eval_result = evaluator.evaluate_response(response=response)
    eval_result.passing


In [14]:
if "components" in external_exercises:
    install_packages(*['-U', 'llama-index-callbacks-arize-phoenix'])


In [15]:
if "components" in external_exercises:
    import llama_index
    import os

    PHOENIX_API_KEY = "<PHOENIX_API_KEY>"
    os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"api_key={PHOENIX_API_KEY}"
    llama_index.core.set_global_handler(
        "arize_phoenix",
        endpoint="https://llamatrace.com/v1/traces"
    )


### Llamada a una herramienta

Ejecuta la función de clima del curso con sus datos de ejemplo y valida los argumentos mediante su firma.

In [16]:
def get_weather(location: str) -> str:
    """Useful for getting the weather for a given location."""
    print(f"Getting weather for {location}")
    return f"The weather in {location} is sunny"

from inspect import signature

tool_registry = {"my_weather_tool": get_weather}
tool_request = {"name": "my_weather_tool", "arguments": {"location": "New York"}}
function = tool_registry[tool_request["name"]]
bound = signature(function).bind(**tool_request["arguments"])
weather_result = function(*bound.args, **bound.kwargs)
assert weather_result == "The weather in New York is sunny"
print(weather_result)


Getting weather for New York
The weather in New York is sunny


In [17]:
if "tools" in external_exercises:
    from llama_index.core.tools import FunctionTool

    def get_weather(location: str) -> str:
        """Useful for getting the weather for a given location."""
        print(f"Getting weather for {location}")
        return f"The weather in {location} is sunny"

    tool = FunctionTool.from_defaults(
        get_weather,
        name="my_weather_tool",
        description="Useful for getting the weather for a given location.",
    )
    tool.call("New York")


In [18]:
if "tools" in external_exercises:
    from llama_index.core import VectorStoreIndex
    from llama_index.core.tools import QueryEngineTool
    from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
    from llama_index.vector_stores.chroma import ChromaVectorStore

    embed_model = HuggingFaceEmbedding("BAAI/bge-small-en-v1.5")

    db = chromadb.PersistentClient(path="./alfred_chroma_db")
    chroma_collection = db.get_or_create_collection("alfred")
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

    index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)

    llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")
    query_engine = index.as_query_engine(llm=llm)
    tool = QueryEngineTool.from_defaults(query_engine, name="some useful name", description="some useful description")


In [19]:
if "gmail" in external_exercises:
    install_packages(*['llama-index-tools-google'])


In [20]:
if "gmail" in external_exercises:
    from llama_index.tools.google import GmailToolSpec

    tool_spec = GmailToolSpec()
    tool_spec_list = tool_spec.to_tool_list()


In [21]:
if "gmail" in external_exercises:
    [(tool.metadata.name, tool.metadata.description) for tool in tool_spec_list]


In [22]:
if "mcp" in external_exercises:
    install_packages(*['llama-index-tools-mcp'])


In [23]:
if "mcp" in external_exercises:
    from llama_index.tools.mcp import BasicMCPClient, McpToolSpec

    # We consider there is a mcp server running on 127.0.0.1:8000, or you can use the mcp client to connect to your own mcp server.
    mcp_client = BasicMCPClient("http://127.0.0.1:8000/sse")
    mcp_tool = McpToolSpec(client=mcp_client)

    # get the agent
    agent = await get_agent(mcp_tool)

    # create the agent context
    agent_context = Context(agent)


### Agente con contexto

Ejecuta la multiplicación del curso y conserva el nombre de un turno para el siguiente.

In [24]:
def multiply(a: int, b: int) -> int:
    """Multiplies two integers and returns the resulting integer"""
    return a * b

def run_local_agent(message, context):
    if message.startswith("Mi nombre es "):
        context["name"] = message.removeprefix("Mi nombre es ").rstrip(".")
        return f"Hola, {context['name']}."
    if message == "¿Cómo me llamo?":
        return context.get("name", "Todavía no tengo tu nombre.")
    if message == "¿Cuánto es 2 por 2?":
        return multiply(2, 2)
    raise ValueError("Este ejercicio solo admite los mensajes de ejemplo.")

context = {}
assert run_local_agent("¿Cuánto es 2 por 2?", context) == 4
print(run_local_agent("Mi nombre es Bob.", context))
answer = run_local_agent("¿Cómo me llamo?", context)
assert answer == "Bob"
print("Nombre recuperado:", answer)


Hola, Bob.
Nombre recuperado: Bob


In [25]:
if "agents" in external_exercises:
    from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
    from llama_index.core.agent.workflow import AgentWorkflow
    from llama_index.core.tools import FunctionTool

    # define sample Tool -- type annotations, function names, and docstrings, are all included in parsed schemas!
    def multiply(a: int, b: int) -> int:
        """Multiplies two integers and returns the resulting integer"""
        return a * b

    # initialize llm
    llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

    # initialize agent
    agent = AgentWorkflow.from_tools_or_functions(
        [FunctionTool.from_defaults(multiply)],
        llm=llm
    )


In [26]:
if "agents" in external_exercises:
    # stateless
    response = await agent.run("What is 2 times 2?")

    # remembering state
    from llama_index.core.workflow import Context

    ctx = Context(agent)

    response = await agent.run("My name is Bob.", ctx=ctx)
    response = await agent.run("What was my name again?", ctx=ctx)


In [27]:
if "agents" in external_exercises:
    from llama_index.core.tools import QueryEngineTool

    query_engine = index.as_query_engine(llm=llm, similarity_top_k=3) # as shown in the Components in LlamaIndex section

    query_engine_tool = QueryEngineTool.from_defaults(
        query_engine=query_engine,
        name="name",
        description="a specific description",
        return_direct=False,
    )
    query_engine_agent = AgentWorkflow.from_tools_or_functions(
        [query_engine_tool],
        llm=llm,
        system_prompt="You are a helpful assistant that has access to a database containing persona descriptions. "
    )


In [28]:
if "agents" in external_exercises:
    from llama_index.core.agent.workflow import (
        AgentWorkflow,
        FunctionAgent,
        ReActAgent,
    )

    # Define some tools
    def add(a: int, b: int) -> int:
        """Add two numbers."""
        return a + b


    def subtract(a: int, b: int) -> int:
        """Subtract two numbers."""
        return a - b


    # Create agent configs
    # NOTE: we can use FunctionAgent or ReActAgent here.
    # FunctionAgent works for LLMs with a function calling API.
    # ReActAgent works for any LLM.
    calculator_agent = ReActAgent(
        name="calculator",
        description="Performs basic arithmetic operations",
        system_prompt="You are a calculator assistant. Use your tools for any math operation.",
        tools=[add, subtract],
        llm=llm,
    )

    query_agent = ReActAgent(
        name="info_lookup",
        description="Looks up information about XYZ",
        system_prompt="Use your tool to query a RAG system to answer information about XYZ",
        tools=[query_engine_tool],
        llm=llm
    )

    # Create and run the workflow
    agent = AgentWorkflow(
        agents=[calculator_agent, query_agent], root_agent="calculator"
    )

    # Run the system
    response = await agent.run(user_msg="Can you add 5 and 3?")


### Flujo con eventos y estado

El flujo recibe una solicitud, recupera sus documentos y conserva un contador de pasos.

In [29]:
from dataclasses import dataclass

@dataclass
class Start:
    query: str

@dataclass
class Retrieved:
    query: str
    matches: list

@dataclass
class Stop:
    result: str

async def retrieve_step(event, state):
    state["steps"] += 1
    return Retrieved(event.query, retrieve(event.query))

async def answer_step(event, state):
    state["steps"] += 1
    if not event.matches:
        return Stop("No se encontró una fuente.")
    node = event.matches[0]
    return Stop(f"{node['text']} [{node['source']}]")

async def run_workflow(query):
    state = {"steps": 0}
    retrieved = await retrieve_step(Start(query), state)
    result = await answer_step(retrieved, state)
    return result, state

result, state = await run_workflow("cargador")
assert result.result.endswith("[manual.txt]") and state["steps"] == 2
empty_result, _ = await run_workflow("astronomía")
assert empty_result.result == "No se encontró una fuente."
print(result.result)
print("Pasos ejecutados:", state["steps"])


El teléfono requiere un cargador USB-C [manual.txt]
Pasos ejecutados: 2


In [30]:
if "workflow" in external_exercises:
    install_packages(*['llama-index-utils-workflow'])


In [31]:
if "workflow" in external_exercises:
    from llama_index.core.workflow import StartEvent, StopEvent, Workflow, step

    class MyWorkflow(Workflow):
        @step
        async def my_step(self, ev: StartEvent) -> StopEvent:
            # do something here
            return StopEvent(result="Hello, world!")


    w = MyWorkflow(timeout=10, verbose=False)
    result = await w.run()


In [32]:
if "workflow" in external_exercises:
    from llama_index.core.workflow import Event

    class ProcessingEvent(Event):
        intermediate_result: str

    class MultiStepWorkflow(Workflow):
        @step
        async def step_one(self, ev: StartEvent) -> ProcessingEvent:
            # Process initial data
            return ProcessingEvent(intermediate_result="Step 1 complete")

        @step
        async def step_two(self, ev: ProcessingEvent) -> StopEvent:
            # Use the intermediate result
            final_result = f"Finished processing: {ev.intermediate_result}"
            return StopEvent(result=final_result)

    w = MultiStepWorkflow(timeout=10, verbose=False)
    result = await w.run()
    result


In [33]:
if "workflow" in external_exercises:
    from llama_index.core.workflow import Event
    import random


    class ProcessingEvent(Event):
        intermediate_result: str


    class LoopEvent(Event):
        loop_output: str


    class MultiStepWorkflow(Workflow):
        @step
        async def step_one(self, ev: StartEvent | LoopEvent) -> ProcessingEvent | LoopEvent:
            if random.randint(0, 1) == 0:
                print("Bad thing happened")
                return LoopEvent(loop_output="Back to step one.")
            else:
                print("Good thing happened")
                return ProcessingEvent(intermediate_result="First step complete.")

        @step
        async def step_two(self, ev: ProcessingEvent) -> StopEvent:
            # Use the intermediate result
            final_result = f"Finished processing: {ev.intermediate_result}"
            return StopEvent(result=final_result)


    w = MultiStepWorkflow(verbose=False)
    result = await w.run()
    result


In [34]:
if "workflow" in external_exercises:
    from llama_index.utils.workflow import draw_all_possible_flows

    draw_all_possible_flows(w, "flow.html")


In [35]:
if "workflow" in external_exercises:
    from llama_index.core.workflow import Context, StartEvent, StopEvent


    @step
    async def query(self, ctx: Context, ev: StartEvent) -> StopEvent:
        # store query in the context
        await ctx.store.set("query", "What is the capital of France?")

        # do something with context and event
        val = ...

        # retrieve query from the context
        query = await ctx.store.get("query")

        return StopEvent(result=val)


In [36]:
if "agent_workflow" in external_exercises:
    from llama_index.core.agent.workflow import AgentWorkflow, ReActAgent
    from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

    # Define some tools
    def add(a: int, b: int) -> int:
        """Add two numbers."""
        return a + b

    def multiply(a: int, b: int) -> int:
        """Multiply two numbers."""
        return a * b

    llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

    # we can pass functions directly without FunctionTool -- the fn/docstring are parsed for the name/description
    multiply_agent = ReActAgent(
        name="multiply_agent",
        description="Is able to multiply two integers",
        system_prompt="A helpful assistant that can use a tool to multiply numbers.",
        tools=[multiply],
        llm=llm,
    )

    addition_agent = ReActAgent(
        name="add_agent",
        description="Is able to add two integers",
        system_prompt="A helpful assistant that can use a tool to add numbers.",
        tools=[add],
        llm=llm,
    )

    # Create the workflow
    workflow = AgentWorkflow(
        agents=[multiply_agent, addition_agent],
        root_agent="multiply_agent",
    )

    # Run the system
    response = await workflow.run(user_msg="Can you add 5 and 3?")


In [37]:
if "agent_workflow" in external_exercises:
    from llama_index.core.workflow import Context

    # Define some tools
    async def add(ctx: Context, a: int, b: int) -> int:
        """Add two numbers."""
        # update our count
        cur_state = await ctx.store.get("state")
        cur_state["num_fn_calls"] += 1
        await ctx.store.set("state", cur_state)

        return a + b

    async def multiply(ctx: Context, a: int, b: int) -> int:
        """Multiply two numbers."""
        # update our count
        cur_state = await ctx.store.get("state")
        cur_state["num_fn_calls"] += 1
        await ctx.store.set("state", cur_state)

        return a * b

    ...

    workflow = AgentWorkflow(
        agents=[multiply_agent, addition_agent],
        root_agent="multiply_agent",
        initial_state={"num_fn_calls": 0},
        state_prompt="Current state: {state}. User message: {msg}",
    )

    # run the workflow with context
    ctx = Context(workflow)
    response = await workflow.run(user_msg="Can you add 5 and 3?", ctx=ctx)

    # pull out and inspect the state
    state = await ctx.store.get("state")
    print(state["num_fn_calls"])
